# BKinD on JHMDB (split 1, clean) — Kaggle

Trains BKinD on clean JHMDB and scores it (Step 1 of the experiment plan).

**Setup (once):**
1. Upload `JHMDB_rgb.zip`, `joint_positions.zip` and `splits.zip` as one **private** Kaggle dataset.
2. In this notebook: **Add Input** → your dataset.
3. **Settings:** Accelerator = **GPU T4 x2** (recommended over P100, whose old GPU type recent PyTorch builds may not support); **Internet = On** (needed to download the code, packages and ImageNet weights).
4. If the GitHub repo is private: **Add-ons → Secrets**, add `GITHUB_TOKEN` (a GitHub token with read access to the repo).

**Run it in the background:** *Save Version → Save & Run All (Commit)*. It keeps running after you close the browser (up to 12 h), and everything in `/kaggle/working/` is saved as this notebook's output.

**Resuming a cut-off run:** add the previous version's output as an input (**Add Input → Your Work → this notebook**). The training cell finds `checkpoint.pth.tar` there and continues from the last saved epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---- Settings ----
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
RUN_NAME = 'JHMDB_split1_clean'
RUN_DIR = f'/kaggle/working/runs/{RUN_NAME}'   # saved as notebook output
DATA = '/kaggle/temp'                           # scratch space, not saved
import os
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(DATA, exist_ok=True)

In [ ]:
# Get the code (uses the GITHUB_TOKEN secret if the repo is private).
import subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    print('no GITHUB_TOKEN secret; cloning as a public repo')
CODE = f'{DATA}/noisykeypoints'
if not os.path.exists(CODE):
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
!git -C {CODE} log -1 --format='code version: %h %s'

In [ ]:
# Kaggle already has torch, torchvision, scipy, sklearn, opencv, h5py, pyyaml, pandas, seaborn.
!pip install -q piq easydict

In [ ]:
# Find the JHMDB pieces under /kaggle/input (Kaggle usually unzips uploads; plain zips are unzipped here).
import glob, zipfile

def find_dir(test):
    for root, dirs, files in os.walk('/kaggle/input'):
        if test(root, dirs, files):
            return root

def find_or_unzip(name, test):
    d = find_dir(test)
    if d is None:
        z = glob.glob(f'/kaggle/input/**/{name}.zip', recursive=True)
        assert z, f'could not find {name} (folder or zip) under /kaggle/input'
        out = f'{DATA}/unzipped/{name}'
        zipfile.ZipFile(z[0]).extractall(out)
        d = next(r for r, ds, fs in os.walk(out) if test(r, ds, fs))
    print(f'{name}: {d}')
    return d

# frames: a folder holding the 21 action folders, e.g. brush_hair/<clip>/image_00001.jpg
frames = find_or_unzip('JHMDB_rgb', lambda r, ds, fs: 'brush_hair' in ds and
                       os.path.exists(os.path.join(r, 'brush_hair', sorted(os.listdir(os.path.join(r, 'brush_hair')))[0], 'image_00001.jpg')))
# joints: a folder holding action folders with <clip>/joint_positions.mat
joints = find_or_unzip('joint_positions', lambda r, ds, fs: 'brush_hair' in ds and
                       glob.glob(os.path.join(r, 'brush_hair', '*', 'joint_positions.mat')))
# splits: a folder with brush_hair_test_split1.txt
splits = find_or_unzip('splits', lambda r, ds, fs: 'brush_hair_test_split1.txt' in fs)

# Assemble the layout prepare_jhmdb_bkind.py expects, using symlinks (no copying).
J = f'{DATA}/JHMDB'
os.makedirs(J, exist_ok=True)
for name, src in [('frames', frames), ('joint_positions', joints), ('splits', splits)]:
    if not os.path.exists(f'{J}/{name}'):
        os.symlink(src, f'{J}/{name}')

In [ ]:
# Build BKinD's folder layout (train/ and val/ for split 1). /kaggle/input is read-only and on
# another disk, so frames are copied instead of hard-linked (~400 MB, a couple of minutes).
if not os.path.exists(f'{DATA}/JHMDB_bkind/split1/clips.json'):
    !python {CODE}/scripts/prepare_jhmdb_bkind.py --jhmdb {J} --out {DATA}/JHMDB_bkind --split 1
else:
    print('data already prepared')

In [ ]:
# Make a Kaggle copy of the JHMDB config: data path, checkpoint folder, batch 32
# (BKinD's setting at 128 px) when the GPU has enough memory, and all GPUs.
import yaml, torch
BK = f'{CODE}/third_party/BKinD'
cfg = yaml.safe_load(open(f'{BK}/config/JHMDB.yaml'))
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
n_gpu = torch.cuda.device_count()
# gpu=None makes BKinD wrap the model in DataParallel and split each batch across all GPUs
# (batch_size is the total, e.g. 32 = 16 per GPU on T4 x2).
cfg.update(data=f'{DATA}/JHMDB_bkind/split1', checkpoint=RUN_DIR,
           batch_size=32 if gpu_gb >= 14 else 16, workers=4,
           gpu=None if n_gpu > 1 else 0)
yaml.safe_dump(cfg, open(f'{BK}/config/JHMDB_kaggle.yaml', 'w'))
print(f'{n_gpu} x {torch.cuda.get_device_name(0)}, {gpu_gb:.1f} GB each -> '
      f'batch_size {cfg["batch_size"]} total, gpu={cfg["gpu"]}')

In [ ]:
# Train. If a previous version's output is attached as input, resume from its checkpoint.
import shutil
ckpt = f'{RUN_DIR}/checkpoint.pth.tar'
if not os.path.exists(ckpt):
    prev = glob.glob(f'/kaggle/input/**/runs/{RUN_NAME}/checkpoint.pth.tar', recursive=True)
    if prev:
        prev_dir = os.path.dirname(prev[0])
        shutil.copytree(prev_dir, RUN_DIR, dirs_exist_ok=True)   # keeps log.txt and samples too
        print('copied previous run from', prev_dir)
resume = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
print('resuming from ' + ckpt if resume else 'starting fresh')
%cd {BK}
!PYTHONUNBUFFERED=1 python train_video.py --config config/JHMDB_kaggle.yaml {resume} 2>&1 | grep --line-buffered -v "Warning" | tee -a {RUN_DIR}/train_log.txt

In [ ]:
# Training curve: learning rate, train loss, val loss per epoch.
!cat {RUN_DIR}/log.txt

In [ ]:
# Score the trained model on the val (official test) clips.
%cd {CODE}
!python scripts/eval_bkind_jhmdb.py --checkpoint {RUN_DIR}/checkpoint.pth.tar \
    --data {DATA}/JHMDB_bkind/split1 --jhmdb {J} --out {RUN_DIR}/eval

In [ ]:
from IPython.display import Image, display
display(Image(f'{RUN_DIR}/eval/vis_val.png'))